In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

os.makedirs('../report_graphs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

# 1. Konfigurasi Parameter (Sama persis dengan ResNet)
IMG_WIDTH, IMG_HEIGHT = 300, 300
BATCH_SIZE = 32
EPOCHS = 30
TRAIN_DIR = '../dataset/casting_data/train/'
TEST_DIR = '../dataset/casting_data/test/'

# 2. Data Preprocessing
train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=0.1,
    height_shift_range=0.1, horizontal_flip=True, vertical_flip=True, zoom_range=0.1
)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical', color_mode='rgb'
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE, class_mode='categorical', color_mode='rgb', shuffle=False
)

# 3. Membangun Arsitektur Xception
base_model_xception = Xception(weights='imagenet', include_top=False, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3))

for layer in base_model_xception.layers:
    layer.trainable = False

x = base_model_xception.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(2, activation='softmax')(x)

model_xception = Model(inputs=base_model_xception.input, outputs=predictions)

# 4. Kompilasi Model
model_xception.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy', metrics=['accuracy']
)

# 5. Callbacks
callbacks_list = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint('../models/xception_best.h5', monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
]

# 6. Eksekusi Pelatihan
history_xception = model_xception.fit(
    train_generator, epochs=EPOCHS,
    validation_data=test_generator, callbacks=callbacks_list
)

# 7. Evaluasi & Visualisasi Plot
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history_xception.history['accuracy'], label='Train Accuracy')
plt.plot(history_xception.history['val_accuracy'], label='Validation Accuracy')
plt.title('Xception Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_xception.history['loss'], label='Train Loss')
plt.plot(history_xception.history['val_loss'], label='Validation Loss')
plt.title('Xception Loss')
plt.legend()

plt.savefig('../report_graphs/04_learning_curve_xception.png', dpi=300, bbox_inches='tight')
plt.show()

# 8. Tampilkan Matriks Standar
Y_pred = model_xception.predict(test_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = test_generator.classes

print('Classification Report Xception:')
print(classification_report(y_true, y_pred, target_names=test_generator.class_indices.keys()))